# Advanced Usages of `viewer3d` library
##### Notebook author: Jason C. Klima

### Imports

In [ ]:
import ipywidgets
import pyrosetta
import viewer3d as v3d

from bokeh.palettes import Reds256
from IPython.display import display
from pyrosetta.rosetta.protocols.minimization_packing import MinMover
from pyrosetta.rosetta.protocols.simple_moves import SmallMover

try:
    %load_ext jupyter_black
except ImportError:
    pass

### Set visualization backend used throughout the Jupyter notebook:
0) `"py3Dmol"`

1) `"nglview"`

2) `"pymol"`

### Set visualization window size (width, height) used throughout the Jupyter notebook

In [ ]:
backend = "py3Dmol"
window_size = [1200, 600]


def on_backend_change(change):
    global backend
    if change["name"] == "value":
        backend = change["new"]


def on_width_change(change):
    global window_size
    if change["name"] == "value":
        window_size[0] = int(change["new"])


def on_height_change(change):
    global window_size
    if change["name"] == "value":
        window_size[1] = int(change["new"])


dropdown = ipywidgets.Dropdown(
    options=["py3Dmol", "nglview", "pymol"],
    value="py3Dmol",
    description="backend",
)
width_box = ipywidgets.IntText(
    value=1200,
    description="width",
)
height_box = ipywidgets.IntText(
    value=600,
    description="height",
)
dropdown.observe(on_backend_change)
width_box.observe(on_width_change)
height_box.observe(on_height_change)

display(dropdown, width_box, height_box)

### Initialize PyRosetta

In [ ]:
pyrosetta.init(
    options="-out:level 0",
    extra_options="",
    set_logging_handler="logging",
    silent=True,
)

### Visualize AlphaFold pLDDT

In [ ]:
poses = [
    v3d.pose_from_alphafold_id(
        "AF-0000000000004321", version="v2"
    ),  # Uncharacterized protein
    v3d.pose_from_alphafold_id(
        "AF-0000000078834380"
    ),  # Homodimer of Catabolite gene activator family protein
]
v = v3d.alphaFoldPLDDT(
    poses,
    cartoon_color="black",
    rescale=False,
    window_size=window_size,
    continuous_update=True,
    backend=backend,
)
v.show()

### Visualize ligands and metals

In [ ]:
pdb_id = "3DGL"  # 1.8 A Crystal Structure of a Non-biological Protein with Bound ATP in a Novel Bent Conformation
v = v3d.ligandsAndMetals(
    v3d.pose_from_pdb_id(pdb_id),
    window_size=window_size,
    backend=backend,
    modules=[v3d.setBackgroundColor(color="#dcdfe7")],
    gui=True,
)
v.show()

### Interactively visualize the core, boundary, and surface of a protein

In [ ]:
pdb_id = "6X9Z"  # De novo design of transmembrane beta-barrels
v = v3d.coreBoundarySurface(
    v3d.pose_from_pdb_id(pdb_id),
    colorschemes=("redCarbon", "yellowCarbon", "blueCarbon"),
    cartoon_color="black",
    window_size=window_size,
    continuous_update=True,
    backend=backend,
)
v.show()

### Interactively visualize a parametrically designed helical bundle

In [ ]:
v = v3d.makeBundle(
    modules=[],
    aa="THR",
    num_helices=6,
    backend=backend,
    window_size=window_size,
    continuous_update=True,
)
v.show()

### Visualize per-residue clashes

In [ ]:
poses = [
    v3d.pose_from_alphafold_id(
        "AF-H6Z3B6-F1", version="v6"
    ),  # Cytochrome c oxidase subunit 1
    v3d.pose_from_alphafold_id(
        "AF-A0A485P7I0-F1", version="v6"
    ),  # Actin-related protein 2 3 complex subunit 1a
]
# Apply `SmallMover` to create clashes to visualize
small_mover = SmallMover()
small_mover.nmoves(100)
for pose in poses:
    small_mover.apply(pose)
v = v3d.perResidueClashMetric(
    poses,
    vmin=0,
    vmax=10,
    log=None,
    palette=list(reversed(Reds256)),
    window_size=window_size,
    backend=backend,
)
v.show()

### Visualize per-residue energy

In [ ]:
poses = [
    v3d.pose_from_alphafold_id(
        "AF-H9CNK5-F1", version="v6"
    ),  # Lipocalin-like toxin
    v3d.pose_from_alphafold_id(
        "AF-A0A0K0F5V6-F1", version="v6"
    ),  # Glutamate decarboxylase 1 (inferred by orthology to a human protein)
]
# Apply the `MinMover` to minimize in a Rosetta energy function
scorefxn = pyrosetta.create_score_function("corrections_conway2016")
min_mover = MinMover()
min_mover.score_function(scorefxn)
min_mover.min_type("dfpmin_armijo")
for pose in poses:
    min_mover.apply(pose)
v = v3d.perResidueEnergyMetric(
    poses,
    scorefxn=scorefxn,
    vmin=-6,
    vmax=0,
    log=None,
    palette=v3d.get_matplotlib_cmap("viridis"),
    window_size=window_size,
    backend=backend,
)
v.show()

### Visualize per-residue SASA

In [ ]:
pdb_id = "6EGC"  # Single-chain version of 2L4HC2_23 (PDB 5J0K)
v = v3d.perResidueSasaMetric(
    poses=v3d.pose_from_pdb_id(pdb_id),
    mode=0,
    vmin=None,
    vmax=None,
    log=None,
    palette=v3d.get_matplotlib_cmap("YlGnBu_r"),
    window_size=window_size,
    backend=backend,
)
v.show()

### Visualize residues with unsatisfied backbone amine and backbone carbonyl hydrogen bonds

In [ ]:
alphafold_id = "AF-Q91386-F1"  # Growth hormone
v = v3d.unsatSelector(
    pose=v3d.pose_from_alphafold_id(alphafold_id, version="v6"),
    scorefxn=pyrosetta.create_score_function("corrections_conway2016"),
    hbond_energy_cutoff=-0.3,
    backend=backend,
)
v.show()

### Visualize per-residue energy, clashes, SASA, unsatisfied hydrogen bonds, and ligands and metals with the `rosettaViewer`

In [ ]:
poses = [
    v3d.pose_from_pdb_id(
        "7S5B"
    ),  # Unbound State of a De novo designed Protein Binder to the Human Interleukin-7 Receptor
    v3d.pose_from_pdb_id(
        "6WVS"
    ),  # Hyperstable de novo TIM barrel variant DeNovoTIM15
    v3d.pose_from_pdb_id("6REK"),  # Crystal structure of Pizza6-SH with Cu2+
]
v = v3d.rosettaViewer(
    packed_and_poses_and_pdbs=poses,
    window_size=window_size,
    continuous_update=True,
    backend=backend,
)
v.show()